# Creación de Transformers y Pipelines

El siguiente cuaderno presentará la creación de Transformers y Pipelines. Al igual que en cuadernos anteriores, se trabajará con el dataset **NSL-KDD**, una versión mejorada de KDD’99 diseñada para evaluar sistemas de detección de intrusos (IDS) en tráfico de red.  

De forma general, NSL-KDD contiene registros de conexiones etiquetadas como **tráfico normal** o **tipos de ataque**, por lo que es útil como conjunto de referencia para tareas de clasificación, análisis exploratorio y comparación de modelos.

Aunque no representa por completo las redes reales actuales, NSL-KDD sigue siendo ampliamente utilizado en investigación por su estructura, tamaño manejable y etiquetas estandarizadas.

### Fuente del dataset
El siguiente dataset puede descargarse en este enlace: **https://www.kaggle.com/datasets/hassan06/nslkdd**. 

## Import de librerías

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


#### Función auxiliar

In [5]:
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

## Lectura de datos

In [6]:
df = pd.read_csv("data/KDDTrain+.csv")

In [7]:
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20


### División del dataset

In [8]:
train_set, val_set, test_set = train_val_test_split(df, stratify='protocol_type')

train_set.shape, val_set.shape, test_set.shape

((75583, 43), (25195, 43), (25195, 43))

## 1. Construcción de transformers personalizados

In [9]:
# Separar características y etiquetas, se elimina la columna "class" de las características y se guarda como etiqueta
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [10]:
# Reemplazar valores atípicos en las columnas "src_bytes" y "dst_bytes" con NaN
X_train.loc[(X_train["src_bytes"]>400) & (X_train["src_bytes"]<800), "src_bytes"] = np.nan
X_train.loc[(X_train["dst_bytes"]>500) & (X_train["dst_bytes"]<2000), "dst_bytes"] = np.nan

In [11]:
X_train

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64559,0,tcp,systat,S0,0.0,0.0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20
67272,0,tcp,http,SF,210.0,NaN,0,0,0,0,...,255,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21
32452,3,tcp,smtp,SF,889.0,328.0,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21
112657,0,tcp,http,SF,284.0,444.0,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21


### Transformers para atributos numéricos

In [12]:
class DeleteNanRows(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return X.dropna()

In [13]:
# Se elimina las filas con valores nulos utilizando el transformador personalizado
delete_nan = DeleteNanRows()
X_train_prep = delete_nan.fit_transform(X_train)

In [14]:
X_train_prep

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.0,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.0,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.0,0.0,0.0,16
98007,0,udp,domain_u,SF,46.0,139.0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.0,0.0,0.0,20
16447,0,tcp,smtp,SF,1790.0,363.0,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.00,0.0,0.0,0.0,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90665,0,tcp,ftp_data,S0,0.0,0.0,0,0,0,0,...,63,0.25,0.02,0.02,0.00,1.00,1.0,0.0,0.0,18
64559,0,tcp,systat,S0,0.0,0.0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.0,0.0,0.0,20
32452,3,tcp,smtp,SF,889.0,328.0,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.0,0.0,0.0,21
112657,0,tcp,http,SF,284.0,444.0,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,21


In [15]:
# Transofrmador diseñado para escalar de manera sencilla únicamente unas columnas seleccionadas
class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self, attributes):
        self.attributes = attributes
    def fit(self, X, y=None):
        return self  # nothing else to do
    def transform(self, X, y=None):
        X_copy = X.copy()
        scale_attrs = X_copy[self.attributes]
        robust_scaler = RobustScaler()
        X_scaled = robust_scaler.fit_transform(scale_attrs)
        X_scaled = pd.DataFrame(X_scaled, columns=self.attributes, index=X_copy.index)
        for attr in self.attributes:
            X_copy[attr] = X_scaled[attr]
        return X_copy

In [16]:
# Se escala únicamente las columnas "src_bytes" y "dst_bytes" utilizando el transformador personalizado
custom_scaler = CustomScaler(["src_bytes", "dst_bytes"])
X_train_prep = custom_scaler.fit_transform(X_train_prep)

In [17]:
X_train_prep.head(10)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
31899,0,tcp,private,S0,-0.034632,0.000000,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.0,1.0,0.0,0.0,21
89913,0,tcp,private,S0,-0.034632,0.000000,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.0,1.0,0.0,0.0,21
106319,0,icmp,eco_i,SF,0.000000,0.000000,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.0,0.0,0.0,0.0,16
98007,0,udp,domain_u,SF,0.164502,0.448387,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.0,0.0,0.0,0.0,20
16447,0,tcp,smtp,SF,7.714286,1.170968,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.0,0.0,0.0,0.0,21
28800,0,tcp,ftp_data,SF,1.411255,0.000000,0,0,0,0,...,28,1.00,0.00,1.00,0.11,0.0,0.0,0.0,0.0,11
78082,0,tcp,ftp_data,SF,44.939394,0.000000,0,0,0,0,...,51,0.23,0.03,0.23,0.04,0.0,0.0,0.0,0.0,21
69315,0,tcp,systat,S0,-0.034632,0.000000,0,0,0,0,...,5,0.02,0.07,0.00,0.00,1.0,1.0,0.0,0.0,20
100360,0,tcp,private,S0,-0.034632,0.000000,0,0,0,0,...,13,0.05,0.07,0.00,0.00,1.0,1.0,0.0,0.0,21
29208,0,tcp,http,SF,1.419913,13.816129,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,21


In [21]:
# Transormador para codificar únicamente las columnas categoricas y devolver un DataFrame
class CustomOneHotEncoding(BaseEstimator, TransformerMixin):
    def __init__(self):
        self._oh = OneHotEncoder(sparse=False)
        self._columns = None
    def fit(self, X, y=None):
        X_cat = X.select_dtypes(include=['object'])
        self._columns = pd.get_dummies(X_cat).columns
        self._oh.fit(X_cat)
        return self
    def transform(self, X, y=None):
        X_copy = X.copy()
        X_cat = X_copy.select_dtypes(include=['object'])
        X_num = X_copy.select_dtypes(exclude=['object'])
        X_cat_oh = self._oh.transform(X_cat)
        X_cat_oh = pd.DataFrame(X_cat_oh, 
                                columns=self._columns, 
                                index=X_copy.index)
        X_copy.drop(list(X_cat), axis=1, inplace=True)
        return X_copy.join(X_cat_oh)

## 2. Construcción de pipelines

Las pipelines sirven para organizar en un solo flujo todos los pasos de preparación de datos. Esto hace más fácil aplicar el mismo proceso a distintos conjuntos sin repetir código.

Puntos clave:
* Se construyen con pares **(nombre, estimador)**.
* Todos los pasos, excepto el último, deben transformar datos.
* El pipeline ofrece los métodos del **último estimador**.
* Al ejecutar `fit()`, cada paso aplica `fit_transform()` en cadena y pasa su salida al siguiente; el último paso solo ajusta con `fit()`.

In [24]:
# Se construye una pipeline para las columnas numéricas que incluye el paso de imputación y escalado
num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('rbst_scaler', RobustScaler()),
    ])

In [ ]:
# La clase imputer no admite valores categoricos, por lo que se quitan los atributos categoricos
X_train_num = X_train.select_dtypes(exclude=['object'])

X_train_prep = num_pipeline.fit_transform(X_train_num)
X_train_prep = pd.DataFrame(X_train_prep, columns=X_train_num.columns, index=X_train_num.index)

In [27]:
X_train_num.head(10)

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,NaN,53508.0,0,0,0,0,0,1,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,0.0,0.0,0,0,0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,304.0,NaN,0,0,0,0,0,1,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,0.0,0.0,0,0,0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,8.0,0.0,0,0,0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
98007,0,46.0,139.0,0,0,0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.00,0.0,0.0,20
16447,0,1790.0,363.0,0,0,0,0,0,1,0,...,137,0.55,0.04,0.01,0.01,0.00,0.00,0.0,0.0,21
64957,1,NaN,329.0,0,0,0,0,0,1,0,...,181,0.65,0.03,0.01,0.01,0.02,0.02,0.0,0.0,21
100052,0,206.0,NaN,0,0,0,0,0,1,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
28800,0,334.0,0.0,0,0,0,0,0,1,0,...,28,1.00,0.00,1.00,0.11,0.00,0.00,0.0,0.0,11


In [28]:
X_train_prep.head(10)

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0.0,0.000000,235.718062,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,1.833333,1.5,0.00,0.00,0.0,0.0,0.333333
31899,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.236735,-0.515789,0.285714,0.000000,0.0,1.00,1.00,0.0,0.0,0.333333
108116,0.0,1.056680,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,0.500000,3.0,0.00,0.00,0.0,0.0,0.333333
89913,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.191837,-0.473684,0.571429,0.000000,0.0,1.00,1.00,0.0,0.0,0.333333
106319,0.0,-0.141700,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.224490,0.515789,-0.428571,16.666667,28.5,0.00,0.00,0.0,0.0,-1.333333
98007,0.0,0.012146,0.612335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.783673,0.515789,-0.285714,0.000000,0.0,0.00,0.00,0.0,0.0,0.000000
16447,0.0,7.072874,1.599119,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.306122,0.042105,0.142857,0.166667,0.5,0.00,0.00,0.0,0.0,0.333333
64957,1.0,0.000000,1.449339,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.485714,0.147368,0.000000,0.166667,0.5,0.02,0.02,0.0,0.0,0.333333
100052,0.0,0.659919,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,0.000000,0.0,0.00,0.00,0.0,0.0,0.333333
28800,0.0,1.178138,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.138776,0.515789,-0.428571,16.666667,5.5,0.00,0.00,0.0,0.0,-3.000000


In [29]:
# Se utiliza el método ColumnTransformer ejecutar los pipelines de manera conjunta, 
# aplicando cada pipeline a las columnas correspondientes. El resultado es un array 
# de valores numéricos y categóricos transformados.
num_attribs = list(X_train.select_dtypes(exclude=['object']))
cat_attribs = list(X_train.select_dtypes(include=['object']))

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

In [30]:
X_train_prep = full_pipeline.fit_transform(X_train)

In [31]:
X_train_prep = pd.DataFrame(X_train_prep, columns=list(pd.get_dummies(X_train)), index=X_train.index)

In [32]:
X_train_prep.head(10)

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
113467,0.0,0.000000,235.718062,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
31899,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
108116,0.0,1.056680,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
89913,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
106319,0.0,-0.141700,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
98007,0.0,0.012146,0.612335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
16447,0.0,7.072874,1.599119,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
64957,1.0,0.000000,1.449339,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
100052,0.0,0.659919,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
28800,0.0,1.178138,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [33]:
X_train.head(10)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
98007,0,udp,domain_u,SF,46.0,139.0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.00,0.0,0.0,20
16447,0,tcp,smtp,SF,1790.0,363.0,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.00,0.00,0.0,0.0,21
64957,1,tcp,smtp,SF,NaN,329.0,0,0,0,0,...,181,0.65,0.03,0.01,0.01,0.02,0.02,0.0,0.0,21
100052,0,tcp,http,SF,206.0,NaN,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
28800,0,tcp,ftp_data,SF,334.0,0.0,0,0,0,0,...,28,1.00,0.00,1.00,0.11,0.00,0.00,0.0,0.0,11


`X_train_prep` ya está listo para entrenar un modelo; las variables numéricas fueron limpiadas/escaladas y las categóricas se convirtieron en columnas binarias.  